In [48]:
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Import required models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

# Metrics
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)

In [49]:
# Split the data set
df = pd.read_csv("data.csv")

X = df.drop(columns = ['diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Dump the test data to be used in streamlit
pd.concat([X_test, y_test], axis=1).to_csv("test_data.csv", index=False)

In [50]:
# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Dump the scaler to be used in streamlit
os.makedirs("model", exist_ok=True)
joblib.dump(scaler, "model/scaler.pkl")

['model/scaler.pkl']

In [51]:
# Create the model directory
models = {
    "logistic_regression": LogisticRegression(),
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "knn": KNeighborsClassifier(),
    "naive_bayes": GaussianNB(),
    "svm": CalibratedClassifierCV(SVC(random_state=42), ensemble=False)
}

for name, model in models.items():
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Save the model to be used in streamlit
    joblib.dump(model, f"model/{name}.pkl")
# end of training

# Compare the models
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    # accuracy
    acc = round(accuracy_score(y_test, y_pred), 4)

    # AUC
    auc = round(roc_auc_score(y_test, y_proba), 4)

    # precision
    precision = round(precision_score(y_test, y_pred), 4)

    # recall
    recall = round(recall_score(y_test, y_pred), 4)

    # f1
    f1 = round(f1_score(y_test, y_pred), 4)

    # mcc
    mcc = round(matthews_corrcoef(y_test, y_pred), 4)

    print (f"{name}: accuracy={acc:.2f}, auc={auc:.2f}, precision={precision:.2f}, recall={recall:.2f}, f1={f1:.2f}, mcc={mcc:.2f}");
# end of comparision

logistic_regression: accuracy=0.98, auc=1.00, precision=1.00, recall=0.95, f1=0.98, mcc=0.96
decision_tree: accuracy=0.94, auc=0.93, precision=0.95, recall=0.88, f1=0.91, mcc=0.87
knn: accuracy=0.96, auc=0.98, precision=0.97, recall=0.90, f1=0.94, mcc=0.91
naive_bayes: accuracy=0.92, auc=0.99, precision=0.92, recall=0.86, f1=0.89, mcc=0.83
svm: accuracy=0.98, auc=1.00, precision=0.98, recall=0.98, f1=0.98, mcc=0.96
